# KEGG

Use the mapping already generated for The Seed information.

In [1]:
import requests
import pandas as pd

In [2]:
features_wp = pd.read_csv("../data/example_features_wp.tsv", sep="\t")

In [3]:
features_wp

,feature_id,patric_id,refseq_locus_tag,protein_id,locus_tag,wp_id,product
0,PATRIC.426430.67.JBLSAI010000001.CDS.100434.10...,fig|426430.67.peg.104,ACNFN8_00520,NaN,ACNFN8_RS00520,WP_000572235.1,16S rRNA (cytosine(967)-C(5))-methyltransferas...
1,PATRIC.426430.67.JBLSAI010000001.CDS.101744.10...,fig|426430.67.peg.105,ACNFN8_00525,NaN,ACNFN8_RS00525,WP_000626897.1,dual-specificity RNA methyltransferase RlmN
2,PATRIC.426430.67.JBLSAI010000001.CDS.102845.10...,fig|426430.67.peg.106,ACNFN8_00530,NaN,ACNFN8_RS00530,WP_000888494.1,protein-serine/threonine phosphatase Stp1
3,PATRIC.426430.67.JBLSAI010000001.CDS.103585.10...,fig|426430.67.peg.107,ACNFN8_00535,NaN,ACNFN8_RS00535,WP_000579564.1,serine/threonine protein kinase Stk1
4,PATRIC.426430.67.JBLSAI010000001.CDS.105807.10...,fig|426430.67.peg.108,ACNFN8_00540,NaN,ACNFN8_RS00540,WP_000847933.1,ribosome small subunit-dependent GTPase A
...,...,...,...,...,...,...,...
2693,PATRIC.426430.67.JBLSAI010000034.CDS.672.851.rev,fig|426430.67.peg.1969,ACNFN8_14400,NaN,ACNFN8_RS14410,WP_172869962.1,phage antirepressor KilAC domain-containing pr...
2694,PATRIC.426430.67.JBLSAI010000036.CDS.137.742.rev,fig|426430.67.peg.1972,ACNFN8_14415,NaN,ACNFN8_RS14425,WP_257046850.1,N-acetylglucosaminidase
2695,PATRIC.426430.67.JBLSAI010000042.CDS.196.495.fwd,fig|426430.67.peg.2330,ACNFN8_14470,NaN,ACNFN8_RS14480,WP_000466769.1,DUF2951 domain-containing protein
2696,PATRIC.426430.67.JBLSAI010000042.CDS.3.155.fwd,fig|426430.67.peg.2329,ACNFN8_14465,NaN,ACNFN8_RS14475,WP_353305505.1,XkdX family protein


Load the KEGG pathway information

In [4]:
genome_id = "426430.67"

In [5]:
URL = "https://www.bv-brc.org/api/pathway/"
fields = [
    "patric_id", "feature_id", "refseq_locus_tag", "gene", "product",
    "ec_description", "ec_number", "pathway_class", "pathway_ec", "pathway_id",
    "pathway_name",
]

# page through the results; a stable sort keeps the pages consistent
page_size = 10000
rows, offset = [], 0
while True:
    query = (
        f"eq(genome_id,{genome_id})"
        f"&select({','.join(fields)})"
        "&sort(+id)"
        f"&limit({page_size},{offset})"
    )
    r = requests.get(f"{URL}?{query}", headers={"Accept": "application/json"})
    r.raise_for_status()
    batch = r.json()
    rows.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

pathways = pd.DataFrame(rows, columns=fields)
print(pathways)

                   patric_id  \
0     fig|426430.67.peg.1422   
1     fig|426430.67.peg.1022   
2     fig|426430.67.peg.2344   
3     fig|426430.67.peg.1630   
4     fig|426430.67.peg.1285   
...                      ...   
1260   fig|426430.67.peg.863   
1261   fig|426430.67.peg.602   
1262  fig|426430.67.peg.1099   
1263  fig|426430.67.peg.1902   
1264  fig|426430.67.peg.1262   

                                             feature_id refseq_locus_tag  \
0     PATRIC.426430.67.JBLSAI010000002.CDS.480319.48...     ACNFN8_06075   
1     PATRIC.426430.67.JBLSAI010000002.CDS.87584.887...     ACNFN8_04060   
2     PATRIC.426430.67.JBLSAI010000005.CDS.4621.6150...     ACNFN8_10405   
3     PATRIC.426430.67.JBLSAI010000003.CDS.134575.13...     ACNFN8_06990   
4     PATRIC.426430.67.JBLSAI010000002.CDS.353669.35...     ACNFN8_05390   
...                                                 ...              ...   
1260     PATRIC.426430.67.JBLSAI010000014.CDS.2.121.fwd     ACNFN8_13750   
1261  P

In [7]:
column_order = ["wp_id", "refseq_locus_tag", "gene", "product", "ec_description", "ec_number",
    "pathway_class", "pathway_ec", "pathway_id", "pathway_name",]
final_data = (
    pathways.merge(
        features_wp[["patric_id", "wp_id"]], on="patric_id", how="left"
        )
    .dropna(subset="wp_id")
    .reindex(columns=column_order)
    )


In [8]:
final_data

,wp_id,refseq_locus_tag,gene,product,ec_description,ec_number,pathway_class,pathway_ec,pathway_id,pathway_name
0,WP_001070676.1,ACNFN8_06075,NaN,3-ketoacyl-CoA thiolase (EC 2.3.1.16) @ Acetyl...,Acetyl-CoA C-acyltransferase,2.3.1.16,Lipid Metabolism,00592_2.3.1.16,00592,alpha-Linolenic acid metabolism
1,WP_000777581.1,ACNFN8_04060,NaN,"N-acetyl-L,L-diaminopimelate aminotransferase ...",Transaminases (aminotransferases),2.6.1.-,Amino Acid Metabolism,00360_2.6.1.-,00360,Phenylalanine metabolism
2,WP_000094576.1,ACNFN8_10405,NaN,2-isopropylmalate synthase (EC 2.3.3.13),2-isopropylmalate synthase,2.3.3.13,Amino Acid Metabolism,00290_2.3.3.13,00290,"Valine, leucine and isoleucine biosynthesis"
4,WP_001292247.1,ACNFN8_05390,nagA,N-acetylglucosamine-6-phosphate deacetylase (E...,N-acetylglucosamine-6-phosphate deacetylase,3.5.1.25,Carbohydrate Metabolism,00052_3.5.1.25,00052,Galactose metabolism
5,WP_000813555.1,ACNFN8_07925,NaN,Pyruvate decarboxylase (EC 4.1.1.1); Alpha-ket...,Pyruvate decarboxylase,4.1.1.1,Carbohydrate Metabolism,00010_4.1.1.1,00010,Glycolysis / Gluconeogenesis
...,...,...,...,...,...,...,...,...,...,...
1259,WP_001805237.1,ACNFN8_06310,cysE,Serine acetyltransferase (EC 2.3.1.30),Serine O-acetyltransferase,2.3.1.30,Energy Metabolism,00920_2.3.1.30,00920,Sulfur metabolism
1261,WP_000883645.1,ACNFN8_02995,NaN,Acetyl-coenzyme A carboxyl transferase alpha c...,Acetyl-CoA carboxylase,6.4.1.2,Energy Metabolism,00720_6.4.1.2,00720,Reductive carboxylate cycle (CO2 fixation)
1262,WP_001044214.1,ACNFN8_04460,NaN,Putative O-acetyltransferase SAS0844 (EC 2.3.1.-),Transferring groups other than amino-acyl groups,2.3.1.-,Xenobiotics Biodegradation and Metabolism,00642_2.3.1.-,00642,Ethylbenzene degradation
1263,WP_000337364.1,ACNFN8_08360,NaN,Choloylglycine hydrolase (EC 3.5.1.24),Choloylglycine hydrolase,3.5.1.24,Lipid Metabolism,00121_3.5.1.24,00121,Secondary bile acid biosynthesis


## Add this to the example data

In [9]:
test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")

/var/folders/z3/xb62blvs5xg7hy35ktdz1mdw0000gn/T/ipykernel_23200/911866719.py:1: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")


In [10]:
test_data = test_data.assign(wp_id=test_data["Majority protein IDs"].str.extract(r"(WP_[\d.]+)", expand=False))

In [12]:
test_data = test_data.merge(final_data, how="left", on="wp_id")

In [13]:
test_data

,Majority protein IDs,NWMN neu,NWMN alt,gen symbol,Annotation,TIGRFAM Sub Role,The SEED,wp_id,refseq_locus_tag,gene,product,ec_description,ec_number,pathway_class,pathway_ec,pathway_id,pathway_name
0,WP_000942300.1|NWMN_RS07340|NWMN_1303|,NWMN_RS07340,NWMN_1303,NWMN_1303,ATP-binding cassette domain-containing protei...,0,NaN,WP_000942300.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,WP_000602071.1|NWMN_RS11270|NWMN_1954|,NWMN_RS11270,NWMN_1954,NWMN_1954,ABC-F type ribosomal protection protein[Staph...,0,NaN,WP_000602071.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,WP_000524836.1|NWMN_RS12920|NWMN_2242|,NWMN_RS12920,NWMN_2242,NWMN_2242,ABC transporter ATP-binding protein[Staphyloc...,Other,NaN,WP_000524836.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,WP_001229077.1|NWMN_RS13625|NWMN_2364|,NWMN_RS13625,NWMN_2364,opp1A,staphylopine-dependent metal ABC transporter ...,Cations and iron carrying compounds,"""ABC transporter dipeptide (TC 3.A.1.5.2)""",WP_001229077.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,WP_001172340.1|NWMN_RS14380|NWMN_2506|,NWMN_RS14380,NWMN_2506,NWMN_2506,AMP-binding protein[Staphylococcus aureus sub...,0,"AMP-(fatty) acid ligases, YtcI homolog""",WP_001172340.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,WP_000396077.1|NWMN_RS11560|NWMN_1998|,NWMN_RS11560,NWMN_1998,NWMN_1998,thiaminase II[Staphylococcus aureus subsp. au...,0,"""Thiamin biosynthesis""",WP_000396077.1,ACNFN8_11685,tenA,Thiaminase II (EC 3.5.99.2) involved in salvag...,Aminopyrimidine aminohydrolase,3.5.99.2,Metabolism of Cofactors and Vitamins,00730_3.5.99.2,00730,Thiamine metabolism
404,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405,WP_000594959.1|NWMN_RS11555|NWMN_1997|,NWMN_RS11555,NWMN_1997,thiD,bifunctional hydroxymethylpyrimidine kinase/p...,Thiamine,"""Thiamin biosynthesis""",WP_000594959.1,ACNFN8_11690,thiD,Hydroxymethylpyrimidine kinase (EC 2.7.1.49) @...,Phosphomethylpyrimidine kinase,2.7.4.7,Metabolism of Cofactors and Vitamins,00730_2.7.4.7,00730,Thiamine metabolism
406,WP_000594959.1|NWMN_RS11555|NWMN_1997|,NWMN_RS11555,NWMN_1997,thiD,bifunctional hydroxymethylpyrimidine kinase/p...,Thiamine,"""Thiamin biosynthesis""",WP_000594959.1,ACNFN8_11690,thiD,Hydroxymethylpyrimidine kinase (EC 2.7.1.49) @...,Hydroxymethylpyrimidine kinase,2.7.1.49,Metabolism of Cofactors and Vitamins,00730_2.7.1.49,00730,Thiamine metabolism


In [14]:
test_data.to_excel("../data/example_s_aureus_kegg.xlsx", index=False)